In [193]:
import pandas as pd
import numpy as np
import matplotlib as plt

## DATASET CODE PROVIDED BY INSTRUCTOR

In [194]:
# Instructor Data Generation Script
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
np.random.seed(42)
total_rows = 101000
vendors = np.random.choice([1, 2], size=total_rows)
base_time = datetime(2026, 1, 1)
pickups = [(base_time + timedelta(minutes=int(i * 0.45))).strftime('%Y-%m-%d %H:%M:%S') for i in range(total_rows)]
dropoffs = [(base_time + timedelta(minutes=int(i * 0.45 + 15))).strftime('%Y-%m-%d %H:%M:%S') for i in range(total_rows)]
passengers = np.random.choice([1, 2, 4], size=total_rows, p=[0.8, 0.15,
0.05]).astype(str)
null_indices = np.random.choice(total_rows, 500, replace=False)
for idx in null_indices: passengers[idx] = ""
distances = np.round(np.random.uniform(0.5, 15.0, size=total_rows), 2)
zero_distance_indices = np.random.choice(total_rows, 100, replace=False)
for idx in zero_distance_indices: distances[idx] = 0.0
fares = np.round(2.50 + (distances * 2.50), 2)
tips = np.round(np.where(np.random.rand(total_rows) > 0.4, fares * 0.15, 0.0), 2)
totals = np.round(fares + tips + 0.50, 2)
negative_indices = np.random.choice(total_rows, 50, replace=False)
for idx in negative_indices: totals[idx] = -10.0
df = pd.DataFrame({
'vendor_id': vendors, 'pickup_datetime': pickups, 'dropoff_datetime':
dropoffs,
'passenger_count': passengers, 'trip_distance': distances, 'fare_amount':
fares,
'tip_amount': tips, 'total_amount': totals
})
df.to_csv('nyc_taxi_sample.csv', index=False)
df_clean = df.copy()

## Milestone 1 - Structural Profiling:
- Programmatically ingest your dataset file into a Pandas DataFrame.
- Execute initial profiling methods including .info(), .describe(), and .shape to establish foundational insights regarding row volume, data types, and index distribution

In [195]:
print(df_clean.info())
print(df_clean.head())
print(df_clean.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101000 entries, 0 to 100999
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   vendor_id         101000 non-null  int64  
 1   pickup_datetime   101000 non-null  object 
 2   dropoff_datetime  101000 non-null  object 
 3   passenger_count   101000 non-null  object 
 4   trip_distance     101000 non-null  float64
 5   fare_amount       101000 non-null  float64
 6   tip_amount        101000 non-null  float64
 7   total_amount      101000 non-null  float64
dtypes: float64(4), int64(1), object(3)
memory usage: 6.2+ MB
None
   vendor_id      pickup_datetime     dropoff_datetime passenger_count  \
0          1  2026-01-01 00:00:00  2026-01-01 00:15:00               2   
1          2  2026-01-01 00:00:00  2026-01-01 00:15:00               4   
2          1  2026-01-01 00:00:00  2026-01-01 00:15:00               2   
3          1  2026-01-01 00:01:00  2026-01-01 0

## Milestone 2 - Automated Data Cleansing
- Apply systematic rules to process messy production data logs without altering raw sources. 
- Identify missing value fields across columns and execute explicit drop logic or median value imputations. 
- Cast temporal log objects from unstructured strings into strict pandas datetime64 values, and isolate/remove mathematical anomalies such as negative transaction amounts,
zero trip distances, or empty indices

### identifying and removing/imputing nulls

In [196]:
# checking for any null values in the rows
df_clean.isnull().sum()
# -- result is 0 null values in all rows for this dataset

### STANDARD CODE IN CASE OF NULL ROWS IN DATASET ###
#counting NA rows
df_clean.isna().sum()
#(set 'how=all' to only remove rows that have ALL columns empty as it is not helpful for our analysis)
#(ignoring_index to reindex after dropped)
df_clean = df_clean.dropna(ignore_index=True)

### IF SOME ROWS ARE EMPTY, WE WILL RENAME TO 'Unknown' ###
# goes indexes each column
for i in range(len(df_clean.columns)):
    #sets variable to column name
    column = df_clean.columns[i]
    # goes through each row
    for j in range(len(df_clean[column])):
        # looks for values within column that are empty (or '')
        if df_clean[column][j] == '':
            #sets to unknown
            df_clean[column][j] = 'Unknown'

/var/folders/3y/ldxff8k17wjcyzkq19xw1srm0000gn/T/ipykernel_33038/4027192935.py:22: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_clean[column][j] = 'Unknown'
/var/folders/3y/ldxff8k17wjcyzkq19xw1srm0000gn/T/ipykernel_33038/4027192935.py:2

In [197]:
df_clean['passenger_count'].value_counts()

passenger_count
1          80257
2          15306
4           4937
Unknown      500
Name: count, dtype: int64

In [198]:
### IF SOME ROWS ARE EMPTY, WE WILL RENAME TO 'Unknown' ###
# goes indexes each column
for i in range(len(df_clean.columns)):
    #sets variable to column name
    column = df_clean.columns[i]
    # goes through each row
    for j in range(len(df_clean[column])):
        # looks for values within column that are empty (or '')
        if df_clean[column][j] == '':
            #sets to unknown
            df_clean[column][j] = 'Unknown'

### changing datatypes for all columns

In [199]:
# ignoring errors as some rows are represented as 'Unknown' if empty

# ensuring integer columns are as expected
df_clean['vendor_id'] = df_clean['vendor_id'].astype(int, errors='ignore')
df_clean['passenger_count'] = df_clean['passenger_count'].astype(dtype=int,errors='ignore')

# ensuring date columns are as expected
df_clean['pickup_datetime'] = pd.to_datetime(df_clean['pickup_datetime'],errors='ignore')
df_clean['dropoff_datetime'] = pd.to_datetime(df_clean['dropoff_datetime'],errors='ignore')

# ensuring float columns are as expected
df_clean['trip_distance'] = df_clean['trip_distance'].astype(float,errors='ignore')
df_clean['fare_amount'] = df_clean['fare_amount'].astype(float,errors='ignore')
df_clean['tip_amount'] = df_clean['tip_amount'].astype(float,errors='ignore')
df_clean['total_amount'] = df_clean['total_amount'].astype(float,errors='ignore')

/var/folders/3y/ldxff8k17wjcyzkq19xw1srm0000gn/T/ipykernel_33038/350630776.py:8: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df_clean['pickup_datetime'] = pd.to_datetime(df_clean['pickup_datetime'],errors='ignore')
/var/folders/3y/ldxff8k17wjcyzkq19xw1srm0000gn/T/ipykernel_33038/350630776.py:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df_clean['dropoff_datetime'] = pd.to_datetime(df_clean['dropoff_datetime'],errors='ignore')


### removing nonsensical values
- ex.0 for passenger_count, trip_distance
- keeping negative fare amounts as this might represent refunds

In [ ]:
# makes no sense to have no distance in trip
print('The number rows with trip_distance = 0 is :',len(df_clean[df_clean['trip_distance'] == 0]))
# keeping only rows that have trip_distance > 0
df_clean = df_clean[df_clean['trip_distance'] > 0]
print('The number rows with trip_distance = 0 after cleaning is :',len(df_clean[df_clean['trip_distance'] == 0]))
# doesn't make sense to have no passengers on taxi ride, will clean

#print('The number rows with passenger_count = 0 is :',len(df_clean[df_clean['passenger_count'] == 0]))
#df_clean = df_clean[df_clean['passenger_count'] > 0]

# will not include in cleaning, as it could be a refund on trip
print('The number rows with fare_amount < 0 is :',len(df_clean[df_clean['fare_amount'] < 0]))

The number rows with trip_distance = 0 is : 100
The number rows with trip_distance = 0 after cleaning is : 0
The number rows with fare_amount < 0 is : 0
